# Clase 03 (versión condensada) — Customer journey con pandas

> **Pregunta de negocio:** ¿Cómo se comportan los clientes y dónde los
> perdemos?

Cambiamos de empresa: ahora trabajamos para una tienda online de productos
variados. Su equipo de producto pide:

> "En cuatro meses han entrado a la web más de un millón de visitantes y
> compra una fracción mínima. Queremos saber en qué punto del recorrido se
> nos van, cuánto tardan en decidirse los que compran y si los que vienen una
> vez vuelven. Y queremos tres propuestas concretas de mejora para el próximo
> trimestre."

Los datos son el log de eventos de **Retailrocket**, una tienda online real
que publicó cuatro meses y medio de actividad anonimizada (mayo a septiembre
de 2015): una fila cada vez que un visitante ve un producto, lo añade al
carrito o lo compra. Para responder hay que mirar al cliente como una
**secuencia de eventos en el tiempo** (el *customer journey*) y construir
con ella sesiones, funnels, cohortes y una matriz de transición.

En esta clase no hay ningún modelo de *machine learning*: la herramienta es
**pandas**, y casi todo se resuelve con una misma idea, **agrupar y después
resumir o transformar**. Aprenderás a usarla bien: qué devuelve cada función,
con cuántas filas y qué errores son los habituales.

Este notebook es una **versión condensada** del notebook completo de la
clase (`clase_03.ipynb`): menos teoría y más práctica con pandas. Si quieres
ver cada paso con más calma, con más ejemplos y con los análisis que aquí se
recortan, ve al notebook completo o al capítulo de teoría de la clase 3.

### Antes de empezar: qué necesitas

1. **Python 3 con Jupyter** y las librerías `pandas`, `numpy`, `matplotlib`
   y `pyarrow` (sirve para leer los ficheros `.parquet`). Si te falta alguna,
   instálala desde una celda con `%pip install pyarrow` (cambiando el nombre
   por el de la que falte) y reinicia el kernel.
2. **Un fichero de datos**, que te dará el profesor, copiado en **la misma
   carpeta que este notebook** y sin cambiarle el nombre:

| Fichero | Para qué sirve |
| --- | --- |
| `events.parquet` | El log de eventos de Retailrocket (37 MB): todo el notebook y las celdas de rescate |

### Índice

| Parte | Contenido |
| --- | --- |
| 0 | Configuración |
| 1 | El patrón de pandas: agrupar y después resumir o transformar |
| 2 | Del log a las sesiones (ejercicios 1 y 2) |
| 3 | Funnels con `crosstab` (ejercicios 3 y 4) |
| 4 | Cohortes con `transform` y `unstack` (ejercicio 5) |
| 5 | Transiciones con `shift(-1)` (ejercicio 6) |
| 6 | Del análisis al caso: el resumen del journey |
| 7 | Chuleta y síntesis |

### Cómo trabajar con este notebook

- Cada parte empieza con una **explicación** y un **ejemplo resuelto** con
  pocos eventos de juguete: léelos y ejecútalos antes de los ejercicios.
- Las celdas de ejercicio tienen huecos `COMPLETA` con un comentario
  `# TODO:` que dice qué falta. Si ves `NameError: name 'COMPLETA' is not
  defined`, te falta rellenar un hueco.
- Tras cada ejercicio de código hay una **celda de comprobación**: si
  imprime `✔ Correcto`, vas bien; si no, su mensaje te orienta.
- Las **celdas de rescate** preparan un resultado para que puedas seguir si
  te has atascado: cambia `USAR_RESCATE = False` por `True` y ejecútala.
- Las celdas **Tu respuesta** son para escribir tus conclusiones.
- El log tiene 2,8 millones de filas: muchas celdas tardan unos segundos.
  Espera a que termine (el indicador `[*]` pasa a un número) antes de
  ejecutar la siguiente.

## 0. Configuración

Importamos las librerías, comprobamos que el fichero está en esta carpeta y
fijamos el estilo de las figuras. Aquí no hace falta semilla: nada de lo que
haremos usa números al azar, así que las cifras saldrán siempre iguales.
También definimos `tabla_funnel`, una función que usaremos en las partes 3
y 6 (se explica en la parte 3).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow  # no se usa directamente: pandas la necesita para leer los .parquet

# El fichero de datos tiene que estar en la misma carpeta que este notebook.
FICHEROS_CLASE = ["events.parquet"]
faltan = [fichero for fichero in FICHEROS_CLASE if not Path(fichero).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan estos ficheros en la carpeta {Path.cwd()}: {faltan}. "
                            "Cópialos a mano junto al notebook, sin cambiarles el nombre. "
                            "Si ya están, abre el notebook desde su propia carpeta.")

# Aquí se guarda tu entregable.
DIR_SALIDAS = Path("salidas")
DIR_SALIDAS.mkdir(exist_ok=True)

pd.set_option("display.width", 100)
pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", "{:,.4f}".format)   # 4 decimales en todas las tablas

AZUL, NARANJA, AQUA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRIS,
    "xtick.color": GRIS,
    "ytick.color": GRIS,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
})


def tabla_funnel(llegan):
    """Conversión de etapa, drop-off y conversión total a partir de los que llegan a cada etapa."""
    tabla = pd.DataFrame({"llegan": llegan})
    tabla["conversion_etapa"] = tabla["llegan"] / tabla["llegan"].shift(1)
    tabla["drop_off"] = 1 - tabla["conversion_etapa"]
    tabla["conversion_total"] = tabla["llegan"] / tabla["llegan"].iloc[0]
    return tabla


print("✔ Configuración lista")

✔ Configuración lista


## 1. El patrón de pandas: agrupar y después resumir o transformar

Un log de eventos tiene una fila por cosa que ha pasado: **quién**
(`visitor_id`), **cuándo** (`timestamp`), **qué** (`event`) y **sobre qué**
(`item_id`). Casi todas las preguntas de la clase se responden igual:
agrupando las filas (por visitante, por sesión, por cohorte) y calculando
algo dentro de cada grupo. Lo que cambia es **cuántas filas devuelve** el
cálculo, y conviene saberlo antes de escribirlo:

| Paso | Código | Qué devuelve |
| --- | --- | --- |
| 1. Agrupar | `log.groupby("visitor_id")` | Un objeto que recuerda los grupos. Todavía no calcula nada. |
| 2a. Resumir | `.size()`, `["col"].min()`, `.max()`, `.nunique()`, `.first()`, `.any()`, `.sum()` | **Una fila por grupo**, con el grupo como índice |
| 2b. Transformar | `["col"].diff()`, `.shift()`, `.cumsum()`, `.transform("min")` | **Una fila por cada fila original**, con el mismo índice: se puede guardar como columna nueva |

**Ejemplo resuelto: tres visitantes y seis eventos**, ya ordenados por
visitante y hora. Primero, resumir:

In [2]:
log_juguete = pd.DataFrame({
    "visitor_id": ["A", "A", "A", "B", "B", "C"],
    "timestamp": pd.to_datetime(["2015-06-01 10:00", "2015-06-01 10:04", "2015-06-01 10:10",
                                 "2015-06-01 10:02", "2015-06-01 10:30", "2015-06-01 11:00"]),
    "event": ["view", "view", "addtocart", "view", "view", "view"],
})
grupos = log_juguete.groupby("visitor_id")

print("size(): eventos de cada visitante")
print(grupos.size())
print("\n['timestamp'].min(): primer evento de cada visitante")
print(grupos["timestamp"].min())

size(): eventos de cada visitante
visitor_id
A    3
B    2
C    1
dtype: int64

['timestamp'].min(): primer evento de cada visitante
visitor_id
A   2015-06-01 10:00:00
B   2015-06-01 10:02:00
C   2015-06-01 11:00:00
Name: timestamp, dtype: datetime64[us]


Las dos salidas tienen **tres filas**, una por visitante, y el visitante
pasa a ser el índice. Son tablas nuevas: no se pueden pegar como columna del
log, que tiene seis filas.

Ahora, transformar. Cada resultado tiene **seis filas**, alineadas con las
del log, así que se guardan como columnas. `diff()` resta a cada hora la
anterior **del mismo visitante** y da una duración (un `Timedelta`); para
leerla en minutos, `.dt.total_seconds() / 60`. `shift(-1)` trae el valor de
la fila **siguiente** del mismo visitante. Añadimos también un `diff()`
**sin** `groupby`, para ver qué pasa si se olvida:

In [3]:
log_juguete["primera"] = grupos["timestamp"].transform("min")   # el mínimo, copiado en cada fila
log_juguete["hueco_min"] = grupos["timestamp"].diff().dt.total_seconds() / 60
log_juguete["siguiente"] = grupos["event"].shift(-1)
log_juguete["hueco_sin_groupby"] = log_juguete["timestamp"].diff().dt.total_seconds() / 60
log_juguete

,visitor_id,timestamp,event,primera,hueco_min,siguiente,hueco_sin_groupby
0,A,2015-06-01 10:00:00,view,2015-06-01 10:00:00,NaN,view,NaN
1,A,2015-06-01 10:04:00,view,2015-06-01 10:00:00,4.0000,addtocart,4.0000
2,A,2015-06-01 10:10:00,addtocart,2015-06-01 10:00:00,6.0000,NaN,6.0000
3,B,2015-06-01 10:02:00,view,2015-06-01 10:02:00,NaN,view,-8.0000
4,B,2015-06-01 10:30:00,view,2015-06-01 10:02:00,28.0000,NaN,28.0000
5,C,2015-06-01 11:00:00,view,2015-06-01 11:00:00,NaN,NaN,30.0000


Cómo se lee:

- `transform("min")` calcula lo mismo que `min()` (10:00 para A, 10:02 para
  B, 11:00 para C), pero lo **copia en cada fila** del visitante.
- `hueco_min` es nulo (`NaN`) en el primer evento de cada visitante, porque
  no tiene anterior; y `siguiente` es nulo en el último. Esos nulos son
  información: marcan dónde empieza y dónde acaba cada grupo.
- Sin `groupby`, el primer evento de B sale con un hueco de **−8 minutos**:
  `diff` le ha restado el último evento de A (10:10), que es de otro
  visitante. Y el de C, 30 minutos desde el último de B. Las operaciones que
  miran a la fila de al lado (`diff`, `shift`) **siempre** van detrás de un
  `groupby`, y la tabla tiene que estar **ordenada** por grupo y tiempo.

**Cómo consultar la ayuda.** `help(pd.Series.diff)` o `pd.Series.shift?` en
una celda muestran la documentación (parámetros, qué devuelve y ejemplos), y
la documentación en línea (pandas.pydata.org, buscando el nombre de la
función) tiene más ejemplos. Las fechas y duraciones tienen sus propias
funciones detrás de `.dt`: `serie.dt.weekday`, `serie.dt.normalize()`,
`serie.dt.total_seconds()`, `serie.dt.days`... Las iremos usando.

## 2. Del log a las sesiones

Un visitante puede entrar el lunes por la mañana, volver el lunes por la
tarde y otra vez el jueves: cada visita es una **sesión**. Muchas preguntas
se hacen por sesión ("¿qué proporción de visitas acaba en compra?"), pero
este log, como muchos, **no trae un identificador de sesión**. Se construye
con un **umbral de inactividad**: si entre dos eventos seguidos del mismo
visitante pasan más de 30 minutos (el valor por defecto de Google Analytics),
empieza una sesión nueva.

Con pandas son tres pasos, todos del patrón de la parte 1:

| Paso | Código | Para qué |
| --- | --- | --- |
| 1. Ordenar | `sort_values(["visitor_id", "timestamp"])` | Sin orden, `diff` restaría eventos que no son seguidos |
| 2. Hueco | `groupby("visitor_id")["timestamp"].diff()` | Tiempo desde el evento anterior del mismo visitante |
| 3. Inicio y número | `hueco.isna() \| (hueco > pd.Timedelta(minutes=30))` y `.cumsum()` | Empieza sesión el primer evento del visitante **o** (`\|`) tras más de 30 minutos; la suma acumulada numera las sesiones |

`pd.Timedelta(minutes=30)` es una duración que se puede comparar con los
huecos. La suma acumulada (`cumsum`) cuenta `True` como 1 y `False` como 0:
el contador sube en uno en cada inicio de sesión.

**Ejemplo resuelto: un visitante con ocho eventos en un día**, que llegan
**desordenados**, como pasa cuando un log junta datos de varios servidores:

In [4]:
log_demo = pd.DataFrame({
    "visitor_id": 1,
    "timestamp": pd.to_datetime([
        "2015-06-01 10:50", "2015-06-01 10:00", "2015-06-01 14:00", "2015-06-01 10:05",
        "2015-06-01 14:03", "2015-06-01 10:12", "2015-06-01 11:10", "2015-06-01 11:02"]),
    "event": ["view", "view", "view", "view", "transaction", "addtocart", "addtocart", "view"],
})
# 1. Ordenar. reset_index(drop=True) vuelve a numerar las filas 0, 1, 2... en el nuevo orden.
log_demo = log_demo.sort_values(["visitor_id", "timestamp"]).reset_index(drop=True)
# 2. Hueco con el evento anterior del mismo visitante.
hueco_demo = log_demo.groupby("visitor_id")["timestamp"].diff()
log_demo["hueco_min"] = hueco_demo.dt.total_seconds() / 60
# 3. Inicio de sesión y numeración.
log_demo["inicio"] = hueco_demo.isna() | (hueco_demo > pd.Timedelta(minutes=30))
log_demo["session_id"] = log_demo["inicio"].cumsum()
log_demo

,visitor_id,timestamp,event,hueco_min,inicio,session_id
0,1,2015-06-01 10:00:00,view,NaN,True,1
1,1,2015-06-01 10:05:00,view,5.0000,False,1
2,1,2015-06-01 10:12:00,addtocart,7.0000,False,1
3,1,2015-06-01 10:50:00,view,38.0000,True,2
4,1,2015-06-01 11:02:00,view,12.0000,False,2
5,1,2015-06-01 11:10:00,addtocart,8.0000,False,2
6,1,2015-06-01 14:00:00,view,170.0000,True,3
7,1,2015-06-01 14:03:00,transaction,3.0000,False,3


Dos huecos superan los 30 minutos (38 y 170), así que el día tiene **tres
sesiones**: de 10:00 a 10:12, de 10:50 a 11:10 y de 14:00 a 14:03. La
columna `inicio` vale `True` en las filas 0, 3 y 6, y su suma acumulada da
1, 1, 1, 2, 2, 2, 3, 3. Con un umbral de 60 minutos, el hueco de 38 ya no
cortaría y saldrían dos sesiones: **el umbral es una decisión**, y toda
cifra por sesión se cita con el suyo.

*Para saber más: notebook completo, parte A y B2; capítulo de la clase 3,
apartados 03.3.1 a 03.3.3.*

### Ejercicio 1 · Auditar el log de eventos · completar código + interpretar

Antes de analizar, la calidad: tamaño, visitantes, duplicados exactos y
cuántos eventos tiene cada visitante. El log tiene cinco columnas: las
cuatro de la parte 1 y `transaction_id`, el pedido de cada compra, que no
usaremos. Pistas:

```python
# pd.read_parquet("fichero.parquet") carga un fichero parquet en una tabla.
# tabla.duplicated() marca True las filas idénticas a una anterior; .sum() las cuenta.
# tabla["columna"].nunique() cuenta valores distintos; .value_counts() cuenta filas de cada valor.
```

In [ ]:
# TU CÓDIGO
eventos = pd.read_parquet("events.parquet")  # TODO: carga el fichero events.parquet
print(eventos.shape)
print("Columnas:", eventos.columns.tolist())
print("Periodo:", eventos["timestamp"].min(), "->", eventos["timestamp"].max())
n_visitantes = eventos['visitor_id']  # TODO: número de visitantes distintos
n_duplicados = eventos.duplicated().sum()  # TODO: número de filas duplicadas exactas
por_visitante = eventos.groupby('visitor_id').size()  # TODO: eventos de cada visitante (una fila por visitante)
print(f"Visitantes: {n_visitantes:,}")
print(f"Duplicados exactos: {n_duplicados:,}")
print(f"Eventos por visitante: mediana {por_visitante.median():.0f}, máximo {por_visitante.max():,}")
print(f"Visitantes con un solo evento: {(por_visitante == 1).mean():.1%}")
recuento =   # TODO: eventos de cada tipo
recuento

In [ ]:
# Comprobación
assert eventos.shape == (2756101, 5), "Revisa la carga: el log tiene 2 756 101 filas y 5 columnas."
assert n_visitantes == 1_407_580, "Revisa los visitantes: cuenta valores distintos de visitor_id."
assert n_duplicados == 460, "Revisa los duplicados: cuéntalos antes de eliminarlos (son 460)."
assert len(por_visitante) == n_visitantes and por_visitante.max() == 7757, \
    "por_visitante debe tener una fila por visitante: groupby('visitor_id').size()."
print("✔ Correcto")

Eliminamos los 460 duplicados exactos (mismo visitante, evento, producto y
milisegundo). A partir de aquí, `eventos` es el log limpio:

In [ ]:
eventos = eventos.drop_duplicates()
print(f"Eventos sin duplicados: {len(eventos):,}")

Preguntas: (i) ¿cuántos carritos y cuántas compras hay por cada 100 vistas?
¿Qué adelanta del problema de negocio? (ii) Un visitante tiene 7 757
eventos. ¿Qué puede ser y qué cambia que `visitor_id` sea una cookie del
navegador y no un cliente identificado?

**Tu respuesta:**

### Ejercicio 2 · Sesionizar el log · completar código + interpretar

(a) Convierte los tres pasos del ejemplo en una **función**, `sesionizar`,
que reciba el log y el umbral en minutos y devuelva el log con la columna
`session_id`. La usaremos varias veces, con distintos umbrales. El
`.copy()` del paso 1 hace que la función trabaje sobre una copia y no toque
la tabla que recibe. La comprobación la prueba con un visitante que tiene
eventos a las 9:00, 9:20, 10:05 y 10:10, **desordenados**: calcula antes a
mano cuántas sesiones esperas con 30 y con 60 minutos.

In [ ]:
# TU CÓDIGO
def sesionizar(eventos, umbral_min):
    """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
    # 1. Ordenar (copy: trabajamos sobre una copia y no tocamos la tabla original).
    eventos = COMPLETA  # TODO: ordena por visitante y tiempo, y copia
    # 2. Hueco con el evento anterior del mismo visitante (nulo en su primer evento).
    hueco = COMPLETA  # TODO: hueco con el evento anterior del mismo visitante
    # 3. Inicio de sesión: hueco nulo (isna) o mayor que el umbral; | significa "o".
    nueva = COMPLETA  # TODO: True si empieza sesión
    # cumsum suma los True acumulados: sube en uno en cada inicio de sesión.
    eventos["session_id"] = nueva.cumsum()
    return eventos

In [ ]:
# Comprobación
log_prueba = pd.DataFrame({
    "visitor_id": 7,
    "timestamp": pd.to_datetime(["2015-06-01 09:20", "2015-06-01 10:10",
                                 "2015-06-01 09:00", "2015-06-01 10:05"]),
    "event": "view",
})
for umbral in [30, 60]:
    print(f"Umbral {umbral} min -> {sesionizar(log_prueba, umbral)['session_id'].nunique()} sesiones")
assert sesionizar(log_prueba, 30)["session_id"].nunique() == 2, \
    "Con 30 minutos deben salir 2 sesiones (huecos de 20, 45 y 5 minutos): ¿ordenas por visitante y tiempo?"
assert sesionizar(log_prueba, 60)["session_id"].nunique() == 1, \
    "Con 60 minutos debe salir 1 sesión: ningún hueco supera los 60 minutos."
print("✔ Correcto")

(b) Sesioniza el log real con 30 minutos y cuenta las sesiones. La celda
tarda unos segundos.

In [ ]:
# TU CÓDIGO
eventos = COMPLETA  # TODO: sesioniza el log con un umbral de 30 minutos
n_sesiones = COMPLETA  # TODO: número de sesiones distintas
print(f"Sesiones: {n_sesiones:,}")
sesiones_por_visitante = eventos.groupby("visitor_id")["session_id"].nunique()
print(f"Sesiones por visitante: media {sesiones_por_visitante.mean():.2f}; "
      f"con una sola sesión, el {(sesiones_por_visitante == 1).mean():.1%} de los visitantes")
eventos[["visitor_id", "timestamp", "event", "session_id"]].head(6)

In [ ]:
# Comprobación
assert n_sesiones == 1_761_675, \
    "Deben salir 1 761 675 sesiones. Revisa que ordenas por visitante Y por tiempo."
print("✔ Correcto")

Pregunta: ¿cuántas sesiones tiene de media cada visitante y qué adelanta
sobre si vuelven? ¿Qué pasaría con las sesiones si te olvidaras del
`groupby` en el paso 2?

**Tu respuesta:**

**¿Cuánto importa el umbral?** Como el umbral es una decisión, se comprueba
cuánto cambian las cifras con otros valores (**análisis de sensibilidad**).
La celda dada sesioniza el log con 5, 30 y 120 minutos y calcula el número
de sesiones y la proporción de sesiones con alguna compra. Fíjate en
`ses["event"].eq("transaction").groupby(ses["session_id"]).any()`:
`groupby` acepta también **otra serie** de la misma longitud (aquí, el
número de sesión de cada evento) y agrupa según sus valores; `any()` marca
con `True` cada sesión con al menos una compra (resumir), y la media de esos
`True`/`False` es la proporción. Si no has terminado el ejercicio 2, activa antes la celda de
rescate: define `sesionizar` y crea `eventos` sin duplicados y con las
sesiones de 30 minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

In [ ]:
filas = []
for umbral in [5, 30, 120]:
    ses = sesionizar(eventos[["visitor_id", "timestamp", "event"]], umbral)
    compra = ses["event"].eq("transaction").groupby(ses["session_id"]).any()   # una fila por sesión
    filas.append({"umbral_min": umbral, "sesiones": ses["session_id"].nunique(),
                  "sesiones_con_compra": compra.mean()})
pd.DataFrame(filas).set_index("umbral_min")

De 5 a 120 minutos, las sesiones bajan de 1,93 a 1,70 millones (un 12 %) y
la proporción de sesiones con compra va del 0,88 % al 0,78 %. Las
conclusiones gruesas no cambian, pero una cifra concreta sí, así que se
cita con su umbral: "el 0,81 % de las sesiones (umbral de 30 minutos) tiene
alguna compra".

*Para saber más: notebook completo, ejercicio 3 (con el histograma de los
huecos, que muestra por qué 30 minutos es razonable); capítulo de la clase
3, apartado 03.3.4.*

## 3. Funnels con `crosstab`

Si te has atascado antes, activa la celda de rescate: define `sesionizar` y
crea `eventos` sin duplicados y con las sesiones de 30 minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

**Qué es un funnel, en tres líneas:** el recorrido **vista → carrito →
compra** como una serie de etapas. Para cada etapa se calcula la
**conversión de etapa** (qué proporción de los que llegaron a la anterior
llega a esta), el ***drop-off*** (1 menos esa conversión: los que se
pierden) y la **conversión total** (qué proporción de los que empezaron
llega a esta).

**Las funciones:**

| Función | Qué hace | Cómo la usamos |
| --- | --- | --- |
| `pd.crosstab(filas, columnas)` | Cuenta cuántas filas del log hay de cada combinación | `pd.crosstab(eventos["session_id"], eventos["event"])`: una fila por sesión, una columna por tipo de evento |
| `.gt(0)` | "Mayor que 0": convierte cada recuento en `True`/`False` | `True` si la sesión tiene al menos un evento de ese tipo |
| `&`, `~` | "Y" y "no" entre columnas de `True`/`False` | Funnel **estricto**: una sesión llega al carrito solo si también tiene vista |
| `.sum()` sobre `True`/`False` | Cuenta los `True` | Cuántas sesiones llegan a cada etapa |
| `tabla_funnel(llegan)` | Nuestra función de la configuración | Divide cada etapa entre la anterior (`shift(1)`) y entre la primera (`iloc[0]`) |

**Ejemplo resuelto: diez sesiones escritas como log** (16 eventos). Todas
ven algún producto; cuatro añaden al carrito (S1, S3, S6 y S8) y dos
compran (S1 y S6).

In [ ]:
log_funnel_ejemplo = pd.DataFrame({
    "session_id": list(range(1, 11)) + [1, 3, 6, 8] + [1, 6],
    "event": ["view"] * 10 + ["addtocart"] * 4 + ["transaction"] * 2,
})
por_sesion_ejemplo = pd.crosstab(log_funnel_ejemplo["session_id"],
                                 log_funnel_ejemplo["event"]).gt(0)
print(por_sesion_ejemplo.head(3))
llegan_ejemplo = pd.Series({
    "view": por_sesion_ejemplo["view"].sum(),   # sum() cuenta los True
    "addtocart": (por_sesion_ejemplo["view"] & por_sesion_ejemplo["addtocart"]).sum(),
    "transaction": (por_sesion_ejemplo["view"] & por_sesion_ejemplo["addtocart"]
                    & por_sesion_ejemplo["transaction"]).sum(),
})
tabla_funnel(llegan_ejemplo)

De 10 a 4 sesiones es una conversión del 40 % (*drop-off* del 60 %); de 4 a
2, del 50 %; en total, 2 de 10, un 20 %. La conversión total es el
**producto** de las de etapa (0,4 × 0,5 = 0,2): por eso un funnel se lee
etapa a etapa, para ver dónde está la pérdida.

**La unidad importa.** La misma pregunta, "¿qué proporción compra?", da
cifras distintas por sesión y por visitante. Un visitante que entra tres
veces y compra en la tercera cuenta como 1 compra de 3 sesiones, pero como
1 comprador de 1 visitante. Con `crosstab` solo cambia la columna de las
filas: `session_id` o `visitor_id`. Regla: **numerador y denominador, en la
misma unidad**.

*Para saber más: notebook completo, B3 (incluye las compras que el funnel
estricto deja fuera); capítulo de la clase 3, apartados 03.4.1 a 03.4.4.*

### Ejercicio 3 · El funnel por sesión y por visitante · completar código + interpretar

Completa el funnel estricto con una fila por sesión y después con una fila
por visitante: es el código del ejemplo, con `eventos`. La celda tarda unos
segundos.

In [ ]:
# TU CÓDIGO
por_sesion = COMPLETA  # TODO: True si la sesión tiene algún evento de cada tipo
llegan_sesion = pd.Series({
    "view": por_sesion["view"].sum(),
    "addtocart": COMPLETA,  # TODO: sesiones con vista y carrito
    "transaction": COMPLETA,  # TODO: con las tres etapas
})
por_visitante_ev = COMPLETA  # TODO: lo mismo, una fila por visitante
llegan_visitante = pd.Series({
    "view": por_visitante_ev["view"].sum(),
    "addtocart": COMPLETA,  # TODO: visitantes con vista y carrito
    "transaction": COMPLETA,  # TODO: con las tres etapas
})
# pd.concat con un diccionario apila las dos tablas y usa las claves como primer nivel del índice.
pd.concat({"por sesión": tabla_funnel(llegan_sesion),
           "por visitante": tabla_funnel(llegan_visitante)})

In [ ]:
# Comprobación
assert len(por_sesion) == 1_761_675, "por_sesion debe tener una fila por sesión (1 761 675)."
assert list(llegan_sesion) == [1_755_781, 38_726, 11_143], \
    "Revisa llegan_sesion: cada etapa exige también las anteriores (&)."
assert list(llegan_visitante) == [1_404_179, 34_401, 10_228], \
    "Revisa llegan_visitante: es el mismo funnel con una fila por visitante (visitor_id)."
print("✔ Correcto")

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
etapas = ["vista → carrito", "carrito → compra"]
conv_s = 100 * tabla_funnel(llegan_sesion)["conversion_etapa"].iloc[1:].to_numpy()
conv_v = 100 * tabla_funnel(llegan_visitante)["conversion_etapa"].iloc[1:].to_numpy()
fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(etapas))
ax.barh(y + 0.18, conv_s, height=0.34, color=AZUL, label="por sesión")
ax.barh(y - 0.18, conv_v, height=0.34, color=NARANJA, label="por visitante")
for yy, v in zip(np.r_[y + 0.18, y - 0.18], np.r_[conv_s, conv_v]):
    ax.text(v + 0.5, yy, f"{v:.1f} %".replace(".", ","), va="center", fontsize=9)
ax.set_yticks(y, etapas)
ax.invert_yaxis()
ax.set(xlabel="% de la etapa anterior", xlim=(0, 38), title="Conversión de cada etapa")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

**Dos cifras que conviene no confundir.** El funnel parte de 1 755 781
sesiones con vista, no de las 1 761 675 que hay: 5 894 sesiones no tienen
ninguna vista (empiezan en el carrito o en la compra). Y su conversión total
(0,63 %) es menor que el 0,81 % de sesiones con alguna compra del análisis
de sensibilidad, aunque las dos son "por sesión, 30 minutos": el 0,81 %
cuenta las 14 297 sesiones con cualquier compra entre todas las sesiones; el
funnel **estricto** solo las 11 143 que pasan por vista, carrito y compra,
entre las sesiones con vista. Al citar una, se dice cuál.

Preguntas: (i) ¿en qué etapa se pierden más sesiones en volumen y cuál es
más accionable? Cita las cifras. (ii) ¿Por qué la conversión total por
visitante (0,73 %) es mayor que por sesión (0,63 %)? Un informe divide los
10 228 compradores entre las 1 755 781 sesiones con vista: ¿qué le dices?

**Tu respuesta:**

**Tiempo entre etapas: `groupby` con dos claves y `unstack`.** ¿Cuánto
tardan en decidirse los que compran? Para cada sesión hace falta el momento
de la **primera** vista, el primer carrito y la primera compra:

- `groupby(["session_id", "event"])["timestamp"].min()` resume con **dos
  claves**: una fila por cada combinación sesión × tipo de evento, con un
  índice de dos niveles.
- `.unstack()` pasa el último nivel del índice (el tipo de evento) a
  **columnas**: queda una fila por sesión y una columna por tipo. Si una
  sesión no tiene algún tipo de evento, esa celda queda nula (`NaT`).
- Restar dos columnas de fechas da duraciones; `.dt.total_seconds() / 60`
  las pasa a minutos.

**Ejemplo resuelto:** una sesión con dos vistas, un carrito y una compra.

In [ ]:
una_sesion_ejemplo = pd.DataFrame({
    "session_id": 1,
    "timestamp": pd.to_datetime(["2015-06-01 10:00", "2015-06-01 10:02",
                                 "2015-06-01 10:05", "2015-06-01 10:09"]),
    "event": ["view", "view", "addtocart", "transaction"],
})
resumido = una_sesion_ejemplo.groupby(["session_id", "event"])["timestamp"].min()
print(resumido, end="\n\n")              # índice de dos niveles: (sesión, evento)
primeras_ejemplo = resumido.unstack()   # los eventos pasan a columnas
print(primeras_ejemplo, end="\n\n")
minutos_carrito = (primeras_ejemplo["addtocart"] - primeras_ejemplo["view"]).dt.total_seconds() / 60
print(f"Vista -> carrito: {minutos_carrito.iloc[0]:.0f} min")

La primera vista es la de las 10:00 (no la de las 10:02, porque `min()` se
queda con la más temprana) y el carrito llega 5 minutos después. Las
columnas salen en orden alfabético (`addtocart`, `transaction`, `view`), no
en el del funnel: se seleccionan por nombre.

*Para saber más: notebook completo, ejercicio 6; capítulo de la clase 3,
apartado 03.4.5.*

### Ejercicio 4 · Tiempo entre etapas · completar código + interpretar

Completa `primeras`, quédate con las sesiones que tienen las tres etapas
**en orden** (primera vista ≤ primer carrito ≤ primera compra) y calcula los
minutos de cada paso. Pistas:

```python
# tabla.dropna(subset=["a", "b"]) quita las filas con algún nulo en esas columnas.
# tabla[(tabla["a"] <= tabla["b"]) & (tabla["b"] <= tabla["c"])] exige las dos condiciones a la vez.
```

In [ ]:
# TU CÓDIGO
primeras = COMPLETA  # TODO: primer evento de cada tipo, tipos en columnas
completas = COMPLETA  # TODO: sesiones con las tres etapas
completas = COMPLETA  # TODO: en orden
minutos = pd.DataFrame({
    "vista -> carrito": COMPLETA,  # TODO: minutos de la vista al carrito
    "carrito -> compra": COMPLETA,  # TODO: minutos del carrito a la compra
})
print(f"Sesiones con las tres etapas en orden: {len(completas):,}")
resumen_min = minutos.describe(percentiles=[0.5, 0.9]).T   # .T: una fila por paso
resumen_min["count"] = resumen_min["count"].astype(int)    # el recuento, como número entero
print(resumen_min.to_string(float_format="{:,.1f}".format))

In [ ]:
# Comprobación
assert len(completas) == 10_411, \
    "Deben quedar 10 411 sesiones: con las tres etapas (sin nulos) y en orden temporal."
assert round(minutos["vista -> carrito"].median(), 1) == 3.0, \
    "La mediana de la vista al carrito es 3,0 minutos: ¿restas carrito menos vista y pasas a minutos?"
assert round(minutos["carrito -> compra"].median(), 1) == 4.3, \
    "La mediana del carrito a la compra es 4,3 minutos: ¿restas compra menos carrito?"
print("✔ Correcto")

Son menos que las 11 143 sesiones compradoras del funnel porque aquí se
exige además el **orden** temporal; el funnel solo pedía que la sesión
tuviera las tres etapas.

Preguntas: (i) ¿cuánto tarda la sesión típica en cada paso? ¿Usarías la
media o la mediana en un informe, y por qué? (ii) ¿Ves un cuello de botella
en el pago?

**Tu respuesta:**

## 4. Cohortes con `transform` y `unstack`

Si te has atascado antes, activa la celda de rescate: define `sesionizar` y
crea `eventos` sin duplicados y con las sesiones de 30 minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

**Qué es una tabla de cohortes, en tres líneas:** una **cohorte** agrupa a
los visitantes por la semana de su primera visita. Para cada cohorte se
cuenta cuántos siguen activos 1, 2, 3... semanas después, y se divide por
los de la semana 0: es la **retención**. Así se comparan visitantes **a la
misma edad**, aunque llegaran en meses distintos.

Hacen falta tres columnas en el log y una tabla:

| Paso | Código | Qué hace |
| --- | --- | --- |
| Semana del evento | `ts.dt.normalize() - pd.to_timedelta(ts.dt.weekday, unit="D")` | `normalize()` quita la hora; `weekday` vale 0 el lunes... 6 el domingo; restar esos días lleva al lunes |
| Cohorte | `groupby("visitor_id")["semana"].transform("min")` | La primera semana del visitante, **copiada en todas sus filas** (transformar) |
| Semanas desde el alta | `(semana - cohorte).dt.days // 7` | Días entre las dos fechas, divididos entre 7 sin decimales (`//`) |
| Tabla | `groupby(["cohorte", "semanas_desde_alta"])["visitor_id"].nunique().unstack()` | Visitantes **distintos** por cohorte y semana; las semanas, en columnas |
| Retención | `activos.div(activos[0], axis=0)` | Cada fila entre su semana 0 (`axis=0`: fila a fila) |

**Ejemplo resuelto:** cinco eventos de tres visitantes en mayo de 2015 (el
lunes 4 de mayo es el primer lunes).

In [ ]:
mini_log = pd.DataFrame({
    "visitor_id": ["A", "A", "B", "B", "C"],
    "timestamp": pd.to_datetime(["2015-05-06 18:30", "2015-05-14 09:10", "2015-05-11 12:00",
                                 "2015-05-29 20:45", "2015-05-09 11:00"]),
})
ts = mini_log["timestamp"]
mini_log["semana"] = ts.dt.normalize() - pd.to_timedelta(ts.dt.weekday, unit="D")
mini_log["cohorte"] = mini_log.groupby("visitor_id")["semana"].transform("min")
mini_log["semanas_desde_alta"] = (mini_log["semana"] - mini_log["cohorte"]).dt.days // 7
print(mini_log, end="\n\n")

activos_mini = mini_log.groupby(["cohorte", "semanas_desde_alta"])["visitor_id"].nunique().unstack()
# Recuentos sin decimales (con algún NaN, pandas guarda la columna con decimales).
print(activos_mini.to_string(float_format="{:.0f}".format), end="\n\n")
activos_mini.div(activos_mini[0], axis=0)

Cómo se lee:

- El miércoles 6 de mayo a las 18:30 (`weekday` 2) pasa a las 00:00 del
  lunes 4. Sin `normalize()`, quedaría el lunes 4 a las 18:30, y los eventos
  de una misma semana tendrían "semanas" distintas.
- A y C llegaron la semana del 4 de mayo; A vuelve en la semana 1 y C no:
  la cohorte retiene el 50 % en la semana 1. B llegó la del 11, no vuelve en
  la 1 y sí en la 2: "activo en la semana *k*" **no es acumulativo**.
- Las celdas `NaN` son combinaciones sin ningún visitante. Aquí son un 0 de
  verdad (todas las semanas están observadas), pero en el log real las
  semanas que una cohorte reciente **aún no ha vivido** también salen `NaN`,
  y no son un 0 % de retención. Por eso **no** se usa
  `unstack(fill_value=0)`: el triángulo vacío de la tabla es correcto.

*Para saber más: notebook completo, B4 (incluye cómo leer la tabla por
filas, columnas y diagonales, y la censura al inicio de los datos); capítulo
de la clase 3, apartados 03.5.1 a 03.5.4.*

### Ejercicio 5 · Cohortes semanales y retención · completar código + interpretar

Es el código del ejemplo, con `eventos`. La línea `en_cohortes` (dada) deja
fuera la primera semana del log (solo tiene un domingo) y la última (está
incompleta): las cohortes completas van de la semana del 4 de mayo a la del
7 de septiembre. La celda tarda unos segundos.

In [ ]:
# TU CÓDIGO
ts = eventos["timestamp"]
eventos["semana"] = COMPLETA  # TODO: lunes de la semana, sin hora
eventos["cohorte"] = COMPLETA  # TODO: primera semana del visitante, en todas sus filas
eventos["semanas_desde_alta"] = COMPLETA  # TODO: semanas entre la semana y la cohorte
en_cohortes = eventos[(eventos["cohorte"] >= "2015-05-04") & (eventos["semana"] <= "2015-09-07")]
activos = COMPLETA  # TODO: visitantes distintos por cohorte y semana
retencion = COMPLETA  # TODO: cada fila entre su semana 0
print(f"Cohortes: {len(retencion)}")

In [ ]:
# Comprobación
assert len(retencion) == 19, "Deben salir 19 cohortes. ¿Has quitado la hora con dt.normalize()?"
assert activos.iloc[0, 0] == 71_389, \
    "La cohorte del 4 de mayo tiene 71 389 visitantes en la semana 0: cuenta visitantes distintos (nunique)."
assert round(retencion.iloc[0, 1], 3) == 0.048, "Revisa retencion: cada fila dividida por su semana 0."
assert retencion[12].notna().sum() == 7, \
    "Solo 7 cohortes han llegado a la semana 12: usa unstack() sin fill_value=0 (las demás, vacías)."
print("✔ Correcto")

Las ocho primeras cohortes y semanas, en porcentaje, y la curva media.
Para la curva media se usan **solo las cohortes que ya tienen observada la
semana 12** (7 cohortes): si no, cada punto de la curva se apoyaría en un
grupo distinto de cohortes.

In [ ]:
tabla_ret = (100 * retencion.iloc[:8, :8]).round(1)
tabla_ret.index = tabla_ret.index.strftime("%d/%m")
print(tabla_ret.to_string(float_format="{:.1f}".format))
print(f"\nSemana 1, en las 18 cohortes que ya la tienen: entre {retencion[1].min():.1%} "
      f"y {retencion[1].max():.1%}")

cohortes_12 = retencion[retencion[12].notna()]          # notna: la celda no está vacía
curva_media = cohortes_12.loc[:, 1:12].mean()            # semanas 1 a 12 (con loc, el 12 se incluye)
print(f"Curva media de {len(cohortes_12)} cohortes: semana 1 {curva_media[1]:.2%}, "
      f"semana 4 {curva_media[4]:.2%}, semana 8 {curva_media[8]:.2%}")

# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(curva_media.index, 100 * curva_media.to_numpy(), marker="o", color=AZUL, linewidth=2)
ax.set(title=f"Retención media de las {len(cohortes_12)} cohortes con 12 semanas observadas",
       xlabel="semanas desde la primera visita", ylabel="% de visitantes activos", ylim=(0, None))
fig.tight_layout()
plt.show()

Preguntas: (i) ¿qué proporción de visitantes vuelve la semana siguiente y
cuántos siguen activos a los dos meses? (ii) En la semana 1, la cohorte del
4 de mayo retiene el 4,8 % y la del 22 de junio, el 3,4 %: ¿ha empeorado la
tienda? Ten en cuenta que el log empieza el 3 de mayo, pero la tienda
existía antes. (iii) ¿Qué cambia que `visitor_id` sea una cookie?

**Tu respuesta:**

## 5. Transiciones con `shift(-1)`

Si te has atascado antes, activa la celda de rescate: define `sesionizar` y
crea `eventos` sin duplicados y con las sesiones de 30 minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

**Qué es una matriz de transición, en tres líneas:** cada tipo de evento es
un **estado**, y se cuenta cuántas veces se pasa de cada estado al
siguiente dentro de una sesión, con dos estados artificiales: **inicio**
(antes del primer evento) y **fin** (después del último). Dividida por
filas, cada fila da la proporción de cada estado siguiente: desde un
carrito, ¿qué proporción de veces viene una compra, otra vista o el final de
la sesión? Describe el comportamiento **medio**; no sirve para predecir a un
visitante concreto.

**Las funciones:**

| Función | Qué hace |
| --- | --- |
| `serie.groupby(sesion).shift(-1)` | Trae el evento de la fila **siguiente** de la misma sesión (transformar); el último de cada sesión queda nulo |
| `.fillna("fin")` | Rellena esos nulos con el estado `"fin"` |
| `groupby("session_id")["event"].first()` | El primer evento de cada sesión (resumir): las transiciones desde `"inicio"` |
| `pd.concat([tabla1, tabla2], ignore_index=True)` | Apila dos tablas de pares (desde, hacia) una debajo de otra |
| `pd.crosstab(desde, hacia, normalize="index")` | Cuenta los pares y divide cada fila por su total: la matriz |
| `.reindex(index=[...], columns=[...])` | Ordena filas y columnas en el orden que queramos |

**Ejemplo resuelto: cinco sesiones cortas** (V vista, C carrito, T compra):
VVCT, VV, VC, VCT y VVV, escritas como log, una fila por evento y ya
ordenadas dentro de cada sesión.

In [ ]:
recorridos = ["VVCT", "VV", "VC", "VCT", "VVV"]
log_recorridos_ejemplo = pd.DataFrame(
    [{"session_id": numero, "event": letra}
     for numero, recorrido in enumerate(recorridos, start=1) for letra in recorrido])

estado_ejemplo = log_recorridos_ejemplo["event"]
siguiente_ejemplo = estado_ejemplo.groupby(log_recorridos_ejemplo["session_id"]).shift(-1).fillna("fin")
inicio_ejemplo = log_recorridos_ejemplo.groupby("session_id")["event"].first()
pares_ejemplo = pd.concat([
    pd.DataFrame({"desde": "inicio", "hacia": inicio_ejemplo.to_numpy()}),
    pd.DataFrame({"desde": estado_ejemplo.to_numpy(), "hacia": siguiente_ejemplo.to_numpy()}),
], ignore_index=True)
filas_orden, columnas_orden = ["inicio", "V", "C", "T"], ["V", "C", "T", "fin"]
conteos_ejemplo = pd.crosstab(pares_ejemplo["desde"], pares_ejemplo["hacia"])   # recuentos
print(conteos_ejemplo.reindex(index=filas_orden, columns=columnas_orden), end="\n\n")
matriz_ejemplo = pd.crosstab(pares_ejemplo["desde"], pares_ejemplo["hacia"], normalize="index")
matriz_ejemplo.reindex(index=filas_orden, columns=columnas_orden)

Desde V hay 9 transiciones: 4 hacia otra vista, 3 hacia el carrito y 2
hacia el fin; normalizadas, 0,44, 0,33 y 0,22. Desde un carrito, dos de cada
tres veces se compra y una se termina la sesión. Cada fila suma 1, y la fila
`inicio` dice que las cinco sesiones empiezan con una vista. Los pares que
no aparecen nunca (de C a V, por ejemplo) ya salen con 0 en `crosstab`:
`reindex` solo cambia el orden de filas y columnas.

`.to_numpy()` pasa cada columna a una matriz de NumPy, **sin índice**:
`siguiente_ejemplo` y `estado_ejemplo` tienen el mismo índice, pero
`inicio_ejemplo` tiene como índice la sesión, y al apilar no queremos que
pandas intente alinearlos.

*Para saber más: notebook completo, B5 (incluye los recorridos más
frecuentes y por qué el 71,2 % del funnel y la celda carrito → fin no
coinciden); capítulo de la clase 3, apartados 03.6.1 a 03.6.4.*

### Ejercicio 6 · La matriz de transición real · completar código + interpretar

Es el código del ejemplo, con `eventos`. Antes se ordena por sesión y
tiempo, para que "la fila siguiente" sea el evento siguiente. La celda
tarda unos segundos.

In [ ]:
# TU CÓDIGO
ordenados = eventos.sort_values(["session_id", "timestamp"])
estado = ordenados["event"]
siguiente = COMPLETA  # TODO: evento siguiente de la misma sesión, o "fin"
inicio = COMPLETA  # TODO: primer evento de cada sesión
pares_reales = pd.concat([
    pd.DataFrame({"desde": "inicio", "hacia": inicio.to_numpy()}),
    pd.DataFrame({"desde": estado.to_numpy(), "hacia": siguiente.to_numpy()}),
], ignore_index=True)
matriz = COMPLETA  # TODO: matriz normalizada por filas
matriz = matriz.reindex(index=["inicio", "view", "addtocart", "transaction"],
                        columns=["view", "addtocart", "transaction", "fin"])
print(f"Eventos de carrito: {estado.eq('addtocart').sum():,}")
matriz

In [ ]:
# Comprobación
assert round(matriz.loc["addtocart", "fin"], 4) == 0.2847, \
    "Revisa siguiente: el último evento de cada sesión va seguido de 'fin' (fillna)."
assert round(matriz.loc["inicio", "view"], 4) == 0.9949, "Revisa inicio: el primer evento de cada sesión."
assert np.allclose(matriz.sum(axis=1), 1), "Cada fila debe sumar 1: usa normalize='index'."
print("✔ Correcto")

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
etiquetas = {"inicio": "inicio", "view": "vista", "addtocart": "carrito",
             "transaction": "compra", "fin": "fin"}
fig, ax = plt.subplots(figsize=(6.5, 3.8))
valores_m = 100 * matriz.to_numpy(dtype=float)
ax.imshow(valores_m, cmap="Blues", vmin=0, vmax=110)
for (i, j), v in np.ndenumerate(valores_m):
    ax.text(j, i, f"{v:.1f} %".replace(".", ","), ha="center", va="center", fontsize=10,
            color="white" if v > 60 else "#0b0b0b")
ax.add_patch(plt.Rectangle((3 - 0.5, 2 - 0.5), 1, 1, fill=False, edgecolor=NARANJA, linewidth=3))
ax.set_xticks(range(matriz.shape[1]), [etiquetas[c] for c in matriz.columns])
ax.set_yticks(range(matriz.shape[0]), [etiquetas[c] for c in matriz.index])
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")
ax.set(xlabel="estado siguiente", ylabel="estado actual")
ax.grid(False)
fig.tight_layout()
plt.show()

Preguntas: (i) lee la fila del carrito: ¿qué pasa después de añadir algo?
¿Qué celda es un **callejón sin salida** (intención de compra seguida del
final de la sesión)? (ii) El *drop-off* del carrito en el funnel era del
71,2 % y aquí el carrito → fin es del 28,5 %. ¿Por qué no coinciden?

**Tu respuesta:**

## 6. Del análisis al caso: el resumen del journey

Si te has atascado antes, activa la celda de rescate: crea las cifras de los
ejercicios 3, 5 y 6 que usa esta parte (`llegan_sesion`,
`llegan_visitante`, `matriz` y `curva_media`) a partir de sus recuentos de
referencia, sin volver a calcularlas.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    llegan_sesion = pd.Series({"view": 1_755_781, "addtocart": 38_726, "transaction": 11_143})
    llegan_visitante = pd.Series({"view": 1_404_179, "addtocart": 34_401, "transaction": 10_228})
    # Solo la celda que se usa: 19 636 de los 68 966 eventos de carrito van seguidos de fin.
    matriz = pd.DataFrame({"fin": [19_636 / 68_966]}, index=["addtocart"])
    curva_media = pd.Series({1: 0.037236888940624234})   # semana 1 de la curva media (ejercicio 5)
    print("Rescate cargado: cifras de referencia de los ejercicios 3, 5 y 6")

El equipo de producto necesita pocas cifras, y cada una con **su unidad y
su umbral**: sin ellas, "0,63 %" y "0,73 %" parecerían contradecirse. La
celda dada reúne las seis cifras clave en la tabla `journey` (redondeadas a
cuatro decimales, como proporciones), la compara con las cifras de
referencia y guarda el entregable `salidas/journey.csv`, el mismo que el del
notebook completo.

In [ ]:
journey = pd.DataFrame({
    "metrica": ["vista -> carrito", "carrito -> compra", "conversion total por sesion",
                "conversion total por visitante", "carritos seguidos de fin",
                "retencion semana 1"],
    "unidad": ["sesión", "sesión", "sesión con vista", "visitante con vista",
               "evento de carrito", "visitante, media de 7 cohortes con 12 semanas"],
    "umbral": ["30 min", "30 min", "30 min", "—", "30 min", "—"],
})
funnel_s = tabla_funnel(llegan_sesion)
funnel_v = tabla_funnel(llegan_visitante)
journey["valor"] = [funnel_s.loc["addtocart", "conversion_etapa"],
                    funnel_s.loc["transaction", "conversion_etapa"],
                    funnel_s.loc["transaction", "conversion_total"],
                    funnel_v.loc["transaction", "conversion_total"],
                    matriz.loc["addtocart", "fin"],
                    curva_media[1]]
journey["valor"] = journey["valor"].round(4)
journey.to_csv(DIR_SALIDAS / "journey.csv", index=False)

# Comprobación
referencia_journey = [0.0221, 0.2877, 0.0063, 0.0073, 0.2847, 0.0372]
assert np.allclose(journey["valor"], referencia_journey), \
    "Alguna cifra no cuadra con la referencia: revisa el ejercicio de esa fila."
print("✔ Correcto: guardado en salidas/journey.csv")
journey

**De la observación a la hipótesis.** El funnel dice **dónde** se van los
visitantes, no **por qué**. El abandono del carrito es compatible con muchas
causas (el envío aparece al final y sorprende, el carrito se usa como lista
de deseos, se comparan precios...), y cada una lleva a una acción distinta.
Por eso cada propuesta se escribe como una **hipótesis** en cuatro
eslabones: **observación** (con cifra y unidad) → **posible causa** (una
conjetura) → **acción** → **métrica** que debería moverse, en la misma
unidad que la observación. Se priorizan por impacto y esfuerzo, y solo un
experimento (clase 14) dice si la causa era la buena. Y nunca con *dark
patterns* (urgencia falsa, cargos añadidos en el último paso): una acción
solo es aceptable si funcionaría igual aunque el cliente entendiera
exactamente lo que hacemos.

*Para saber más: notebook completo, B6 y ejercicio 10 (con la tabla de
hipótesis priorizadas, `hipotesis.csv`); capítulo de la clase 3, apartado
03.7.*

Pregunta: escribe una hipótesis completa (los cuatro eslabones) a partir de
una cifra de la tabla `journey`. ¿Aceptarías un mensaje "tu carrito caduca
en 10 minutos" si el carrito no caduca de verdad?

**Tu respuesta:**

## 7. Chuleta y síntesis

### Chuleta de pandas de esta clase

| Para qué | Código | Qué devuelve |
| --- | --- | --- |
| Auditar | `nunique()`, `duplicated().sum()`, `value_counts()`, `drop_duplicates()` | Recuentos; el log sin duplicados |
| Resumir por grupo | `groupby(clave).size()`, `["col"].min()`, `.nunique()`, `.first()`, `.any()` | **Una fila por grupo**, con el grupo como índice |
| Transformar por grupo | `groupby(clave)["col"].diff()`, `.shift(1)`, `.shift(-1)`, `.transform("min")` | **Una fila por fila**, mismo índice: columna nueva |
| Sesionizar | ordenar → `diff()` → `isna() \| (hueco > pd.Timedelta(minutes=30))` → `cumsum()` | Un número de sesión por evento |
| Funnel | `pd.crosstab(unidad, evento).gt(0)`, `&`, `~`, `.sum()` | Unidades que llegan a cada etapa |
| De filas a columnas | `groupby([a, b])[col].min().unstack()` | Una fila por `a`, una columna por `b`; `NaN` donde no hay dato |
| Fechas y duraciones | `.dt.normalize()`, `.dt.weekday`, `.dt.days`, `.dt.total_seconds()`, `pd.to_timedelta(n, unit="D")` | Fechas sin hora, día de la semana, duraciones en días o segundos |
| Retención | `activos.div(activos[0], axis=0)` | Cada fila entre su primera columna |
| Matriz de transición | `pd.crosstab(desde, hacia, normalize="index")` | Cada fila suma 1 |

**Errores frecuentes:**

- Usar `diff` o `shift` sin ordenar o sin `groupby`: se restan eventos de
  otro momento o de otro visitante (los −8 minutos de la parte 1).
- Confundir resumir con transformar: `min()` da una fila por grupo y no se
  puede guardar como columna; para eso, `transform("min")`.
- Mezclar unidades: compradores (visitantes) entre sesiones no es la
  conversión de nada.
- Citar una cifra por sesión sin su umbral.
- Rellenar con `fill_value=0` las semanas que una cohorte aún no ha vivido:
  no son un 0 % de retención.
- Olvidar `dt.normalize()` antes de calcular la semana: las semanas
  conservan la hora y se multiplican.
- Leer la matriz de transición como si predijera a cada visitante: describe
  el comportamiento medio.

### La petición del equipo de producto

Responde en cuatro líneas, con cifras y unidades: dónde se van, cuánto
tardan en decidirse los que compran, si vuelven y qué mejoras propones.

**Tu respuesta:**

### Lo que viene

La clase 4 cambia de señal: de lo que el cliente **hace** a lo que **dice**
(reseñas), y estrena el aprendizaje supervisado con scikit-learn. El
patrón de pandas de hoy (agrupar y después resumir o transformar) seguirá
apareciendo cada vez que haya que preparar los datos de un modelo; la
retención vuelve en la clase 6, para predecir **quién** se va, y las
hipótesis de hoy son lo que un experimento A/B contrasta en la clase 14.